## Домашнее задание 4: Post-training

__Мягкий дедлайн: 08.12.2026 23:59__ \
__Жесткий дедлайн: 10.12.2026 23:59__

### О задании

В этом задании вам предлагается дообучить предобученную LLM, [`Qwen3-0.6B-Base`](https://huggingface.co/Qwen/Qwen3-0.6B-Base), решать задачи по математике. Как принято в стандартном формате пост-обучения, тренировка будет происходить в два этапа: SFT и RL. Обе стадии вам нужно будет реализовать самостоятельно, используя LoRA для дообучения.

Мы расчитываем, что весь цикл обучения должен занять не больше суток на одной T4.

В этом задании __полностью запрещается__ использование агентов.

### Оценки и штрафы

Максимальная оценка за работу — __11.5 баллов__.

Задание должно быть выполнено самостоятельно. «Похожие» решения считаются плагиатом, и все студенты, причастные к этому (включая тех, у кого был совершен плагиат), не могут получить за это более 0 баллов. Весь код должен быть написан самостоятельно. Использование чужого кода запрещено, даже со ссылкой на источник. В разумных пределах, конечно. Пара очевидных строк кода для реализации некоторой незначительной функциональности допустима.

### О данных

Мы будем использовать смесь двух датасетов с математическими задачами: [GSM8K](https://huggingface.co/datasets/openai/gsm8k) (человеческие задачи) и [OpenMathInstruct-2](https://huggingface.co/datasets/nvidia/OpenMathInstruct-2) (сгенерирован Llama-3.1-405B-Instruct в стиле GSM8K). Обработанный и очищенный датасет лежит по ссылке [`ashaba1in/sft_grpo_gsm8k_omi2`](https://huggingface.co/datasets/ashaba1in/sft_grpo_gsm8k_omi2).

| Срез | Размер | Контент |
| --- | --- | --- |
| `sft_train` | 21,000 | смесь GSM8K и OMI-2 |
| `rl_train` | 5,973 | GSM8K без пересечений с SFT |
| `dev` | 500 | GSM8K без пересечений с SFT и RL |
| `test` | 1,319 | Тестовый срез GSM8K |

В колонке `question` лежит вопрос с промптом, объясняющим задачу. В колонке `answer` у `sft_train` лежит подробный ответ с рассуждением, для всех остальных срезов – только численный ответ для подсчета совпадения с предсказанием.

In [3]:
from datasets import load_dataset

dataset = load_dataset("ashaba1in/sft_grpo_gsm8k_omi2")
dataset

DatasetDict({
    sft_train: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 21000
    })
    rl_train: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 5973
    })
    dev: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 1319
    })
})

In [4]:
example = dataset['sft_train'][0]
print('SFT example\n')
print("Question:\n" + example["question"])
print("\nAnswer:\n" + example["answer"])

SFT example

Question:
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: Emily purchased 7 sets of blue pens and 3 sets of green pens. Each set contains 12 pens. How many more blue pens than green pens did Emily buy?
Solution:

Answer:
 Let's break down the number of blue pens and green pens Emily purchased.
For blue pens, she purchased 7 sets. Each set contains 12 pens, so she purchased 7 * 12 = 84 blue pens.
For green pens, she purchased 3 sets. Each set contains 12 pens, so she purchased 3 * 12 = 36 green pens.
Now we find out how many more blue pens than green pens she bought.
The difference is 84 - 36 = 48.
So Emily bought \boxed{48} more blue pens than green pens.


In [5]:
example = dataset['rl_train'][0]
print('RL example\n')
print("Question:\n" + example["question"])
print("\nAnswer:\n" + example["answer"])

RL example

Question:
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: At A.T. Cross Luxury Pens, a pencil sells at twenty-five cents while a pen sells at fifteen cents. Bowen buys 40 pens and 2/5 times more pencils than pens from A.T. Cross Luxury Pens. Calculate the total amount of money Bowen spends.
Solution:

Answer:
20


In [6]:
fewshot_prefix = """
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: The result from the 40-item Statistics exam Marion and Ella took already came out. Ella got 4 incorrect answers while Marion got 6 more than half the score of Ella. What is Marion's score?
Solution: Ella's score is 40 items - 4 items = 36 items.
Half of Ella's score is 36 items / 2 = 18 items.
So, Marion's score is 18 items + 6 items = 24 items.
The final answer is \boxed{24}.

Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: Stephen made 10 round trips up and down a 40,000 foot tall mountain. If he reached 3/4 of the mountain's height on each of his trips, calculate the total distance he covered.
Solution: Up a mountain, Stephen covered 3/4*40000 = 30000 feet.
Coming down, Stephen covered another 30000 feet, making the total distance covered in one round to be 30000+30000 = 60000.
Since Stephen made 10 round trips up and down the mountain, he covered 10*60000 = 600000
The final answer is \boxed{600000}.

"""

### vLLM

[vLLM](https://docs.vllm.ai) — это библиотека для инференса, которая генерирует текст намного быстрее, чем встроенный `model.generate`. Ниже в файле `vllm_utils.py` приведены вспомогательные функции для работы с vLLM.
Чтобы ознакомиться с библиотекой, мы рекомендуем почитать [быстрый старт vLLM](https://docs.vllm.ai/en/stable/getting_started/quickstart/#on-attention-backends) и [примеры использования vLLM](https://docs.vllm.ai/en/stable/examples/). Также можно почитать сам код на [гитхабе](https://github.com/vllm-project/vllm/).

Пайплайн использования:
1. vLLM загружает свою собственную копию замороженной базовой модели в fp16.
2. Адаптер LoRA — единственное, что изменяется во всей модели. Поэтому перед генерацией, его необходимо выгружать с помощью `save_lora_for_vllm` и передавать соответствующий путь в `generate` вместе с __новым__ значением `lora_id`.
3. Для RL используйте в качестве базовой модели объединенную модель SFT (базовая + сдвиг весов от LoRA) и обучайте новый LoRA адаптер.

Практические замечания:
- Долю выделяемой под vLLM памяти можно контролировать с помощью `gpu_memory_utilization`.
- Чтобы получить несколько ответов для каждого запроса, используйте `n=` вместо повторной генерации. Это сэкономит время, так как промпт будет обрабатываться один раз.
- Обучайте модель на `token_ids`, возвращаемых vLLM, а не на повторной токенизации текста, так как новая токенизация может отличаться.
- Для базовой модели передайте `stop=["\nSolve the following math problem."]`, иначе она может продолжить писать новые задачи после ответа.

In [20]:
%%writefile vllm_utils.py

"""
Helper functions fro vLLM with LoRA support

Pipeline:
  * vLLM keeps its own copy of the frozen base weights (fp16) and applies trained LoRA adapter on the fly.
  * Before generation, export the adapter using `save_lora_for_vllm` on disk and
    pass its path to `generate` with a NEW integer `lora_id`.
    vLLM caches adapters by id, so reusing an id keeps generating with this id's weights.
"""

import json
import os

import torch
from safetensors.torch import save_file


def make_llm(
    model_name_or_path: str, gpu_memory_utilization: float = 0.3,
    max_model_len: int = 1024, max_lora_rank: int = 16,
    enable_lora: bool = True, seed: int = 0, enforce_eager: bool = False
):
    """Create a vLLM engine that occupies a specified part of GPU.

    gpu_memory_utilization is the fraction of the whole GPU vLLM can take (weights + KV cache).
    max_lora_rank must be >= trained LoRA rank.
    """
    from vllm import LLM

    return LLM(
        model=model_name_or_path, dtype="float16", gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len, enable_lora=enable_lora, max_lora_rank=max_lora_rank,
        max_loras=1, seed=seed, enforce_eager=enforce_eager
    )


def generate(
    llm, prompts, n: int = 1, temperature: float = 0.0,
    top_p: float = 1.0, max_tokens: int = 512,
    stop=None, seed=None, lora_path=None, lora_id=None
):
    """Sample n completions for every prompt.

    Returns a list (one entry per prompt) of lists (one entry per sample) of dicts:
        {"text": str, "token_ids": list[int], "finish_reason": "stop" | "length"}
    finish_reason == "length" means the completion was truncated at max_tokens.
    Make sure to `token_ids` (not a re-tokenization of `text`) when you compute log-probs for GRPO.
    Pass lora_path=None to sample from the base weights only.
    """
    from vllm import SamplingParams
    from vllm.lora.request import LoRARequest

    if temperature == 0.0:
        assert n == 1, "greedy decoding with n > 1 returns n identical samples"

    params = SamplingParams(
        n=n, temperature=temperature, top_p=top_p,
        max_tokens=max_tokens, stop=stop, seed=seed
    )
    lora_request = None
    if lora_path is not None:
        assert lora_id is not None and lora_id >= 1, "use a new positive integer id for every new adapter"
        lora_request = LoRARequest(f"adapter-{lora_id}", lora_id, lora_path)

    outputs = llm.generate(prompts, params, lora_request=lora_request, use_tqdm=False)
    return [[
        {"text": c.text, "token_ids": list(c.token_ids), "finish_reason": c.finish_reason}
        for c in out.outputs
    ] for out in outputs]


def save_lora_for_vllm(
    lora_weights: dict, out_dir: str, scaling: float, base_model_name_or_path: str = ""
):
    """Export LoRA weights in the adapter format vLLM reads.

    lora_weights: {module_name: (A, B)} for every linear layer you adapted, where
        module_name is the layer's name in model.named_modules(), e.g. "model.layers.0.self_attn.q_proj",
        A has shape [r, in_features] and B has shape [out_features, r],
        so that the adapted layer computes  y = x W^T + scaling * (x A^T) B^T.
    scaling: the factor your LoRA applies to B @ A (usually alpha / r). It is folded into B here.
    """

    os.makedirs(out_dir, exist_ok=True)
    ranks = {tuple(A.shape)[0] for A, _ in lora_weights.values()}
    assert len(ranks) == 1, f"all adapters must have the same rank, got {ranks}"

    r = ranks.pop()
    tensors, targets = {}, set()
    for name, (A, B) in lora_weights.items():
        assert A.dim() == 2 and B.dim() == 2 and A.shape[0] == r and B.shape[1] == r, (name, A.shape, B.shape)
        prefix = f"base_model.model.{name}"
        tensors[f"{prefix}.lora_A.weight"] = A.detach().float().to(torch.float16).cpu().contiguous()
        tensors[f"{prefix}.lora_B.weight"] = (B.detach().float() * scaling).to(torch.float16).cpu().contiguous()
        targets.add(name.split(".")[-1])

    save_file(tensors, os.path.join(out_dir, "adapter_model.safetensors"))
    config = {
        "peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": base_model_name_or_path,
        "r": r, "lora_alpha": r,  # alpha / r = 1 because `scaling` is already folded into B
        "target_modules": sorted(targets), "lora_dropout": 0.0, "bias": "none",
        "fan_in_fan_out": False, "use_rslora": False, "use_dora": False,
        "modules_to_save": None, "inference_mode": True,
    }

    with open(os.path.join(out_dir, "adapter_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    return out_dir

Overwriting vllm_utils.py


In [ ]:
# Run this once to check that vLLM works on your GPU.

from vllm_utils import make_llm, generate, save_lora_for_vllm

llm = make_llm("Qwen/Qwen3-0.6B-Base", gpu_memory_utilization=0.8)

dev = dataset['dev']

outs = generate(
    llm, [fewshot_prefix + ex for ex in dev[:4]['question']],
    n=2, temperature=0.6, top_p=0.95,
    max_tokens=512, stop=["\nSolve the following math problem."])

for question, answer, samples in zip(dev[:4]["question"], dev[:4]["answer"], outs):
    print("Question:\n" + question)
    print("\nGold answer: " + answer)
    print("\nOutputs:")
    for s in samples:
        print(f'  [Finish reason "{s['finish_reason']}", {len(s['token_ids'])} tokens] {s['text'][-150:]!r}')
    print('=' * 100)

# Before every generation:
#   lora = {name: (A, B) for every adapted linear layer}    # A: [r, in_features], B: [out_features, r]
#   save_lora_for_vllm(lora, "adapters/current", scaling=alpha / r)
#   outs = generate(
#       llm, prompts, n=8, temperature=1.0, max_tokens=512,
#       lora_path="adapters/current", lora_id=step + 1
#   )

### Оценка качества

Для оценки качества мы будем генерировать ответы для тестовой выборки с помощью vLLM и оценивать точность предсказаний. Для этого нам необходимо реализовать несколько функций для обработки выходов модели. Они же позже будут использоваться для RL.

__Задание 1 (0.5 балла).__ Реализуйте функционал для обработки ответа.

- `extract_answer(text)`: извлекает содержимое __последнего__ `\boxed{...}`, нормализуя текст внутри: удаляются все лишние фигурные скобки, запятые, знаки доллара и процента (`\%` и `%`), пробелы и точки в конце. Если после нормализации останется число, оно возвращается в формате `float`, иначе возвращается `None`.

- `is_correct(pred, gold)`: проверяет, что `pred` не равен `None` и равен `float(gold)` с точностью до `1e-4`.

- `format_ok(text, finish_reason)`: проверяет, имеет ли вывод правильный формат: генерация должна завершаться по EOS (`finish_reason == "stop"` из vLLM), содержать ровно один `\boxed{`, и `extract_answer` должен возвращать число.

- `is_repetitive(text)`: возвращает `True`, если существуют 10 последовательных слов, которые встречаются 3 или более раз.

- `pass_at_k(n, c, k)`: несмещенная оценка из [Chen et al. (2021)](https://arxiv.org/abs/2107.03374). Если $c$ из $n$ ответов для задачи верны, то $\text{pass@}k = 1 - \binom{n-c}{k} \big/ \binom{n}{k}$, усредненное по задачам. Здесь $n \ge k$.

In [ ]:
# your code here

In [ ]:
# Do not change the tests.

# extract_answer
assert extract_answer("She has 3 + 4 = 7 apples. The final answer is \\boxed{7}.") == 7
assert extract_answer("\\boxed{1,000}") == 1000
assert extract_answer("The answer is \\boxed{\\$42.50}.") == 42.5
assert extract_answer("\\boxed{50\\%}") == 50
assert extract_answer("\\boxed{-3}") == -3
assert extract_answer("\\boxed{ 12 }") == 12
assert extract_answer("First \\boxed{3}, then \\boxed{5}") == 5  # only the last box counts
assert extract_answer("\\boxed{18 \\text{ dollars}}") is None
assert extract_answer("\\boxed{\\frac{1}{2}}") is None           # GSM8K answers are plain numbers
assert extract_answer("The answer is 7.") is None
assert extract_answer("\\boxed{}") is None
assert extract_answer("\\boxed{7") is None

# is_correct
assert is_correct(extract_answer("\\boxed{72}"), "72")
assert is_correct(extract_answer("\\boxed{72.0}"), "72")
assert not is_correct(extract_answer("\\boxed{71}"), "72")
assert not is_correct(None, "72")

# format_ok
assert format_ok("So the total is 7. The final answer is \\boxed{7}.", finish_reason="stop")
assert not format_ok("So the total is 7. The final answer is \\boxed{7}.", finish_reason="length")
assert not format_ok("It is \\boxed{3} or \\boxed{4}.", finish_reason="stop")
assert not format_ok("The answer is 7.", finish_reason="stop")
assert not format_ok("The answer is \\boxed{seven}.", finish_reason="stop")

# is_repetitive
assert is_repetitive("so " * 40)
assert is_repetitive("Then we add 2 to the total and get the new total. " * 3)
assert not is_repetitive("Natalia sold 48 / 2 = 24 clips in May. She sold 48 + 24 = 72 clips in total.")
assert not is_repetitive("Then we add 2 to the total and get the new total. " * 2)

# pass_at_k
assert abs(pass_at_k(n=5, c=2, k=1) - 0.4) < 1e-9
assert abs(pass_at_k(n=10, c=3, k=2) - (1 - 21 / 45)) < 1e-9
assert pass_at_k(n=5, c=0, k=3) == 0.0
assert pass_at_k(n=5, c=4, k=2) == 1.0
print("All Task 1 tests passed")

__Задание 2 (0.5 балла).__ Реализуйте функцию `evaluate(...)`, которая принимает движок vLLM, список задач, префикс подсказки, строки остановки и, при необходимости, LoRA адаптер, и возвращает:

- __жадную точность__ (температура 0): точность ($p$) ответов с 95% [доверительным интервалом](https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval) $p \pm 1.96\sqrt{p(1-p)/N}$.

- __среднее значение pass@1 по 4 ответам__ (температура 0.6, top-p 0.95): средняя точность 4-х ответов на каждую задачу.

- __соответствие формату__, __частоту повторений__, __частоту обрезания__ (`finish_reason == "length"`) и __среднюю длину ответа__ в токенах, на _жадных_ генерациях.

Везде используйте `max_tokens=512`.

In [ ]:
# your code here

__Задание 3 (0.5 балла).__ Оцените базовую модель на тестовых данных `dataset["test"]` в режиме zero-shot и few-shot (префикс `fewshot_prefix`, `stop=["\nSolve the following math problem."]`). Оцените результат. Какие выводы вы можете сделать?

In [ ]:
# your code here

### Supervised Fine-tuning

__Задание 4 (2 балла).__ Обучите LoRA адаптер для `Qwen3-0.6B-Base` на срезе SFT в течение одной эпохи. Во время обучения логируйте лосс, точность на трейне и валидации, соответствие генерации формату и долю повторений в тексте. Сравните результаты SFT модели с базовой из __Задания 3__. Сделайте выводы. Для RL стадии нам требуется модель, следующая формату, но при этом достаточно разнообразная в плане генерации (для RL нужен хороший эксплорейшн). Удалось ли этого достичь?

In [ ]:
# your code here

### Group Relative Policy Optimization

#### Описание метода
Для каждого запроса $q$ мы генирируем группу из $G$ ответов $o_1, \dots, o_G$ с помощью текущей политики и считаем для каждого награду $r_i \in \{0, 1\}$: 1, если ответ правильный и имеет нужный формат, 0 иначе. На основе наград в группе считается advantage:

$$A_i = \frac{r_i - \mathrm{mean}(r_1, \dots, r_G)}{\mathrm{std}(r_1, \dots, r_G)}.$$

В процессе обучения минимизируется следующий функционал 
<!-- Все токены $o_i$ обладают преимуществом $\hat A_i$. При использовании отношения вероятностей $\rho_{i,t} = \pi_\theta(o_{i,t} \mid q, o_{i,<t}) \,/\, \pi_{\theta_\text{old}}(o_{i,t} \mid q, o_{i,<t})$ потери по пакету завершений составляют (обратите внимание на нормализацию в стиле [DAPO](https://arxiv.org/abs/2503.14476)) -->

$$\mathcal{L}(\theta) = -\frac{1}{\sum_i |o_i|} \sum_i \sum_{t=1}^{|o_i|} \Big[ \min\Big(\rho_{i,t} A_i,\; \mathrm{clip}(\rho_{i,t}, 1-\varepsilon, 1+\varepsilon)\, A_i\Big) - \beta D_{i,t} \Big],$$

где $\rho_{i,t} = \pi_\theta(o_{i,t} \mid q, o_{i,<t}) \,/\, \pi_{\theta_\text{old}}(o_{i,t} \mid q, o_{i,<t})$ – отношение оптимизируемой политики и политики, использованной для генерации ответов. $D_{i,t}$ – несмещенная потокенная монте-карло оценка KL дивергенции $\mathrm{KL}(\pi_\theta \,\|\, \pi_\text{ref})$ относительно SFT политики:

$$D_{i,t} = \frac{\pi_\text{ref}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - \log \frac{\pi_\text{ref}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - 1.$$

Заметьте, что мы отходим от оригинальной формулировки [GRPO](https://arxiv.org/pdf/2501.12948) и нормализуем ошибку по общему количеству сгенерированных токенов. Оригинальный GRPO усредняет ошибку для каждого ответа отдельно. [Liu et al. (2025)](https://arxiv.org/abs/2503.20783) показывают, что такая нормализация поощряет излишнее удлинение неправильных ответов.

__Задание 5 (4 балла).__ Реализуйте GRPO и дообучите с его помощью LoRA адаптер для SFT модели на соответствующих данных (`rl_train`). Во время обучения отслеживайте среднюю награду, долю полностью верных и неверных групп, долю ответов в правильном формате, KL дивергенцию, среднюю точность и длину ответа. Проведите сравнение GRPO модели с базовой и SFT по аналогии с предыдущим заданием. Сравните точность ответов при увеличении $k$ в `pass@k` ($k = 1, 2, 4, \dots, 64$). Приведите пример промпта, для которого жадная генерация GRPO дает правильный ответ, а базовая модель – неправильный, и наоборот. Сделайте выводы.

__Важно:__ 
1. Выбранный способ подсчета advantage зануляет градиент для групп со всеми правильными или всеми неправильными ответами. Поэтому для экономии времени выбрасывайте такие группы.
2. RL может разойтись, если брать недостаточно большой батч. Мы рекомендуем выбрать размер группы $G = 8$ и на каждом шаге оптимизироваться минимум по $B = 8$ промптам.
3. Уделите внимание температуре и параметру $\beta$. Подумайте о том, какую роль они играют и какие значения должны принимать.

In [ ]:
# your code here

### Оценка важности составляющих

После того, как у нас получилось рабочее решение, стоит разобраться в том, что для него на самом деле важно. Для этого мы проведем ряд абляций. За каждый пункт можно получить __1 балл__.

1. Обучите GRPO сразу из базовой модели, пропустив SFT. Как такая модификация влияет на скорость сходимости и итоговое качество?
2. Сравните модель обученую с вашим значинием $\beta$ и модель с $\beta = 0$.
3. Попробуйте убрать std нормализацию в advantage, как советуют в [Dr. GRPO](https://arxiv.org/abs/2503.20783). Что изменилось?
4. Обучите LoRA с рангом 1 для GRPO и сравните с вашим выбранным рангом. Стало ли хуже?

In [ ]:
# your code here